# 11 SR Training SeisGAN

Super-resolution training на локальных HR Thebe кубах с on-the-fly degradation.

Базовые классы и функции берутся из `src/seismic_fault_recognition`. Данные должны уже лежать локально; скачивания в ноутбуке нет.

## 1. Bootstrap

Добавляем `src` в `sys.path`, чтобы ноутбук запускался прямо из репозитория.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## 2. Конфигурация эксперимента

Загружаем recipe и YAML-конфиг. Все пути редактируются в `configs/experiments`, а не в коде ноутбука.

In [ ]:
RECIPE_NAME = "sr_training_seisgan"
from seismic_fault_recognition.notebook_utils import (
    dataloader_kwargs,
    ensure_dir,
    load_notebook_context,
    print_context_summary,
    print_path_report,
    seed_everything,
    tuple3,
)

ctx = load_notebook_context(RECIPE_NAME, start=repo_root)
DATA = ctx.data
TRAINING = ctx.training
CHECKPOINTS = ctx.checkpoints
OUTPUTS = ctx.outputs

seed_everything(
    int(ctx.reproducibility.get("seed", TRAINING.get("seed", 42))),
    deterministic=bool(ctx.reproducibility.get("deterministic", True)),
    benchmark=bool(ctx.reproducibility.get("benchmark", False)),
)
print_context_summary(ctx)

## ClearML logging

Создаем ClearML task из recipe/config. Если `clearml` не установлен или отключен в конфиге, объект будет работать как no-op и обучение продолжится локально.

In [ ]:
from seismic_fault_recognition.clearml import (
    clearml_metric_logger,
    init_clearml_from_context,
    report_checkpoint_artifact,
    report_optimizer_lr,
)

clearml_run = init_clearml_from_context(ctx)
print(clearml_run.summary())

## 3. Быстрая проверка путей

Эта ячейка ничего не создает и не скачивает; она только показывает, какие локальные файлы уже доступны.

In [ ]:
print_path_report(
    {
        "train_seis": DATA.get("train_seis", ""),
        "train_fault": DATA.get("train_fault", ""),
        "val_seis": DATA.get("val_seis", ""),
        "val_fault": DATA.get("val_fault", ""),
        "test_seis": DATA.get("test_seis", ""),
        "test_fault": DATA.get("test_fault", ""),
        "output_dir": CHECKPOINTS.get("output_dir", ""),
    },
    ctx.repo_root,
)

## 4. Импорты для этапа

Импортируем только reusable-классы и функции из `src`. В ноутбуке не определяем Dataset/Loss/Model классы.

In [ ]:
import torch
from torch.utils.data import DataLoader

from seismic_fault_recognition.augmentations import build_sr_degradation_pipeline
from seismic_fault_recognition.data import SRDynamicDataset, require_paths
from seismic_fault_recognition.losses import build_loss
from seismic_fault_recognition.models.factory import build_model_by_name
from seismic_fault_recognition.training import save_checkpoint
from seismic_fault_recognition.trainers import train_sr_epoch, validate_sr

## 5. Локальные пути и устройство

Если здесь возникает `FileNotFoundError`, поправьте соответствующий YAML в `configs/experiments`.

In [ ]:
paths = require_paths({"train_seis": DATA["train_seis"], "val_seis": DATA["val_seis"]}, base_dir=ctx.repo_root)
output_dir = ensure_dir(CHECKPOINTS["output_dir"], ctx.repo_root)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patch_size = tuple3(TRAINING["patch_size"])
print(f"device={device}, patch_size={patch_size}")

## 6. Датасеты

Создаем dataset-объекты из базовых классов проекта.

In [ ]:
train_degrade = build_sr_degradation_pipeline(seed=int(TRAINING["seed"]), is_train=True)
val_degrade = build_sr_degradation_pipeline(seed=int(TRAINING["seed"]), is_train=False)
train_ds = SRDynamicDataset(paths["train_seis"], train_degrade, patch_size=patch_size, random_crop=True, seed=int(TRAINING["seed"]))
val_ds = SRDynamicDataset(paths["val_seis"], val_degrade, patch_size=patch_size, random_crop=False, seed=int(TRAINING["seed"]))
train_loader = DataLoader(
    train_ds,
    batch_size=int(TRAINING["batch_size"]),
    **dataloader_kwargs(int(ctx.reproducibility.get("seed", TRAINING["seed"])), int(ctx.reproducibility.get("num_workers", TRAINING["num_workers"])), shuffle=True),
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    **dataloader_kwargs(int(ctx.reproducibility.get("seed", TRAINING["seed"])) + 1, int(ctx.reproducibility.get("num_workers", TRAINING["num_workers"])), shuffle=False),
)
print(f"train samples={len(train_ds)}, val samples={len(val_ds)}")

## 7. DataLoader

Отдельная ячейка для batch size, shuffle и num_workers, чтобы их было легко менять.

In [ ]:
# DataLoader создаются в предыдущей ячейке вместе с SRDynamicDataset.

## 8. Модель, loss, optimizer

Выбор модели и loss берется из recipe/config; веса сохраняются локально в `checkpoints/...`.

In [ ]:
encoder = None
if CHECKPOINTS.get("input"):
    from seismic_fault_recognition.models.omniseis import OmniSeisEncoder
    from seismic_fault_recognition.models.swinunetr_variants import build_swinunetr_tiny
    from seismic_fault_recognition.training import load_checkpoint

    encoder_backbone = build_swinunetr_tiny(
        img_size=tuple3(TRAINING["roi_size"]),
        in_channels=1,
        out_channels=1,
    )
    encoder = OmniSeisEncoder(encoder_backbone)
    encoder_ckpt = require_paths({"encoder_checkpoint": CHECKPOINTS["input"]}, base_dir=ctx.repo_root)["encoder_checkpoint"]
    load_checkpoint(encoder_ckpt, model=encoder, map_location=device, strict=False)

generator = build_model_by_name(ctx.recipe.model_variant, encoder=encoder).to(device)
discriminator = build_model_by_name(
    "patchgan_discriminator_3d",
    base_channels=int(TRAINING.get("discriminator_channels", 64)),
).to(device)
loss_fn = build_loss(ctx.recipe.loss_profile, use_vgg=bool(TRAINING.get("use_vgg", False))).to(device)
optimizer_g = torch.optim.AdamW(generator.parameters(), lr=float(TRAINING["learning_rate"]), weight_decay=1e-5)
optimizer_d = torch.optim.AdamW(discriminator.parameters(), lr=float(TRAINING.get("learning_rate_d", 1e-5)), weight_decay=1e-5)
print(type(generator).__name__, type(discriminator).__name__, type(loss_fn).__name__)


## 9. Обучение

Основной цикл. Он запускает реальные train/validation вызовы и сохраняет latest/best checkpoint.

In [ ]:
train_epoch = clearml_metric_logger(clearml_run, title="Metrics", series_prefix="Train")(train_sr_epoch)
validate_epoch = clearml_metric_logger(clearml_run, title="Metrics", series_prefix="Val")(validate_sr)

best_score = -1.0
for epoch in range(1, int(TRAINING["max_epochs"]) + 1):
    train_metrics = train_epoch(
        generator,
        train_loader,
        optimizer_g,
        loss_fn,
        device=device,
        amp=bool(TRAINING["amp"]),
        discriminator=discriminator,
        optimizer_d=optimizer_d,
        return_metrics=True,
        metric_normalization=ctx.validation.get("sr_metric_normalization", "dataset_stats"),
        metric_data_range=tuple(ctx.validation.get("sr_data_range", [-3.0, 3.0])),
        clearml_iteration=epoch,
    )
    metrics = validate_epoch(
        generator,
        val_loader,
        loss_fn,
        device=device,
        metric_normalization=ctx.validation.get("sr_metric_normalization", "dataset_stats"),
        metric_data_range=tuple(ctx.validation.get("sr_data_range", [-3.0, 3.0])),
        clearml_iteration=epoch,
    )
    report_optimizer_lr(clearml_run, optimizer_g, epoch, series="LR_G")
    report_optimizer_lr(clearml_run, optimizer_d, epoch, series="LR_D")

    combined_metrics = {"train": train_metrics, "val": metrics}
    extra = {
        "discriminator_state_dict": discriminator.state_dict(),
        "optimizer_d_state_dict": optimizer_d.state_dict(),
    }
    latest_path = output_dir / "latest_checkpoint.pth"
    save_checkpoint(latest_path, generator, optimizer=optimizer_g, epoch=epoch, metrics=combined_metrics, extra=extra)

    score = metrics.get("psnr", 0.0) * metrics.get("ssim", 0.0)
    if score > best_score:
        best_score = score
        best_path = output_dir / "best_model.pth"
        save_checkpoint(best_path, generator, optimizer=optimizer_g, epoch=epoch, metrics=combined_metrics, extra=extra)
        report_checkpoint_artifact(clearml_run, "best_sr_model", best_path)
    print(f"epoch={epoch} train={train_metrics} val={metrics}")